# Horizontal Well Geology — TVT Prediction
### CNN + Transformer (PyTorch, GPU)

**Task:** Predict `Target_TVT` (True Vertical Thickness) for horizontal wells using logs,
positional data, and typewell geological reference curves.

**Metric:** RMSE

**Hardware target:** AMD Threadripper (48 cores) / 256GB RAM / RTX PRO 6000 Blackwell (96GB VRAM)

**Why this architecture:** A standard (non-dilated) multi-layer 1D CNN extracts local log
character — bed boundaries, thin markers, curve shape — with strong translation invariance
and cheap compute, then a Transformer self-attention stack integrates that into long-range
structural context across the lateral. This is the "excellent if computational resources
are available" option from the architecture comparison: more parameter-heavy than a plain
TCN because the CNN stage isn't dilated (shallower effective receptive field per layer, so
it needs more layers/width to match TCN's reach), but very strong when compute isn't the
constraint. The typewell's geological reference signal reaches the model as scalar
per-row features (`tvt_geo_offset`, `gr_vs_typewell_diff`, `typewell_GR_ref`,
`typewell_TVT_ref`) rather than through a dedicated cross-attention block.

**Pipeline stages:**
1. Configuration, logging, reproducibility
2. Data loading, memory optimization, missing-value handling
3. Typewell attachment + Geology → integer index (learned embedding)
4. Lightweight feature engineering (positional, distance, geological-offset, gradients)
5. GroupKFold on `WELL_NAME`
6. Sliding-window dataset construction + padding/masking
7. Model: CNN encoder → positional encoding → Transformer self-attention stack → head
8. GPU training loop: AMP, gradient clipping, OneCycle LR, early stopping, checkpointing
9. Optuna hyperparameter search (architecture + optimization), SQLite-backed, resumable, pruned
10. Final CV training, window→row OOF reconstruction, fold/overall RMSE
11. Self-attention visualization, residual analysis, learning curves
12. Model + artifact persistence
13. Inference pipeline + submission generation


## 1. Imports

In [ ]:

# ----------------------------------------------------------------------------
# Core
# ----------------------------------------------------------------------------
import os
import sys
import gc
import json
import time
import math
import random
import pickle
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Dict, Tuple, Any

# ----------------------------------------------------------------------------
# Numerical / data
# ----------------------------------------------------------------------------
import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------------------------------------------------------
# ML / DL
# ----------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import weight_norm
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ----------------------------------------------------------------------------
# Hyperparameter optimization
# ----------------------------------------------------------------------------
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import optuna.visualization as opt_viz

# ----------------------------------------------------------------------------
# Plotting / progress
# ----------------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python           : {sys.version.split()[0]}")
print(f"PyTorch          : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"VRAM             : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"NumPy            : {np.__version__}")
print(f"Pandas           : {pd.__version__}")
print(f"Optuna           : {optuna.__version__}")


## 2. Configuration

In [ ]:

@dataclass
class Config:
    # ------------------------------------------------------------------ paths
    DATA_DIR: Path = Path("../data/rogii-wellbore-geology-prediction/processed_data")

    TRAIN_FILE: str = "train.csv"          # Recommended (faster)
    TYPEWELL_FILE: str = "typewell.csv"
    TEST_FILE: str = "test.csv"

    SAMPLE_SUB_FILE: str = "sample_submission.csv"
    
    OUTPUT_DIR: Path = Path("./outputs_cnn_transformer")
    MODELS_DIR: Path = Path("./outputs_cnn_transformer/models")
    LOGS_DIR: Path = Path("./outputs_cnn_transformer/logs")
    FIGURES_DIR: Path = Path("./outputs_cnn_transformer/figures")
    OPTUNA_DIR: Path = Path("./outputs_cnn_transformer/optuna")
    SUBMISSIONS_DIR: Path = Path("./outputs_cnn_transformer/submissions")
    ATTENTION_DIR: Path = Path("./outputs_cnn_transformer/attention")
    CACHE_DIR: Path = Path("./outputs_cnn_transformer/cache")

    # ------------------------------------------------------------- columns
    GROUP_COL: str = "WELL_NAME"
    TARGET_COL: str = "Target_TVT"
    MD_COL: str = "MD"
    HORIZONTAL_LOG_COLS: Tuple[str, ...] = (
        "ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA", "GR"
    )
    COORD_COLS: Tuple[str, ...] = ("X", "Y", "Z")
    TYPEWELL_TVT_COL: str = "TVT"
    TYPEWELL_GR_COL: str = "GR"
    TYPEWELL_GEOLOGY_COL: str = "Geology"

    # ------------------------------------------------------------- run config
    SEED: int = 42
    N_FOLDS: int = 5
    N_JOBS: int = 48

    # ------------------------------------------------------------- windowing
    WINDOW_SIZE: int = 256
    WINDOW_STRIDE_TRAIN: int = 128
    WINDOW_STRIDE_INFER: int = 64

    # ------------------------------------------------------------- model (architecture-specific defaults; overridden by Optuna)
    CNN_CHANNELS: Tuple[int, ...] = (64, 64, 128, 128)
    CNN_KERNEL_SIZE: int = 5
    CNN_DROPOUT: float = 0.1
    GEOLOGY_EMBED_DIM: int = 8
    D_MODEL: int = 128
    N_TRANSFORMER_LAYERS: int = 3
    N_TRANSFORMER_HEADS: int = 8
    TRANSFORMER_FF_DIM: int = 256
    TRANSFORMER_DROPOUT: float = 0.1
    HEAD_HIDDEN_DIM: int = 128
    HEAD_DROPOUT: float = 0.1

    # ------------------------------------------------------------- training
    BATCH_SIZE: int = 256
    MAX_EPOCHS: int = 100
    LEARNING_RATE: float = 3e-4
    WEIGHT_DECAY: float = 1e-4
    GRAD_CLIP_NORM: float = 1.0
    EARLY_STOPPING_PATIENCE: int = 12
    NUM_WORKERS: int = 16
    USE_AMP: bool = True

    # ------------------------------------------------------------- optuna
    # NOTE: each trial trains a full neural network for several epochs, so this search is
    # deliberately far smaller than the 1000-trial GBM searches.
    N_OPTUNA_TRIALS: int = 150
    OPTUNA_TIMEOUT: Optional[int] = None
    OPTUNA_STUDY_NAME: str = "cnn_transformer_tvt"
    OPTUNA_STORAGE: str = "sqlite:///./outputs_cnn_transformer/optuna/optuna_study.db"
    OPTUNA_N_STARTUP_TRIALS: int = 15
    OPTUNA_CV_FOLDS_FOR_SEARCH: int = 2
    OPTUNA_MAX_EPOCHS: int = 20
    OPTUNA_EARLY_STOPPING_PATIENCE: int = 5
    OPTUNA_CHECKPOINT_EVERY: int = 5

    # ------------------------------------------------------------- final train
    FINAL_MAX_EPOCHS: int = 150
    FINAL_EARLY_STOPPING_PATIENCE: int = 15

    # ------------------------------------------------------------- memory
    REDUCE_MEMORY: bool = True

    def make_dirs(self):
        for d in [self.OUTPUT_DIR, self.MODELS_DIR, self.LOGS_DIR, self.FIGURES_DIR,
                  self.OPTUNA_DIR, self.SUBMISSIONS_DIR, self.ATTENTION_DIR, self.CACHE_DIR]:
            d.mkdir(parents=True, exist_ok=True)


CFG = Config()
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))


## 3. Logging Setup

In [ ]:

import logging

def setup_logger(name: str = "well_geology_dl", log_dir: Path = CFG.LOGS_DIR,
                  level: int = logging.INFO) -> logging.Logger:
    '''Configure a logger that writes to both console and a timestamped file.'''
    log_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = log_dir / f"{name}_{timestamp}.log"

    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.handlers.clear()

    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(sh)
    logger.propagate = False

    logger.info(f"Logger initialized. Log file: {log_path}")
    return logger


logger = setup_logger()
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
logger.info(f"RUN_ID = {RUN_ID}")


## 4. Directory Creation & Reproducibility

In [ ]:

CFG.make_dirs()
logger.info("Directory tree created:")
for d in [CFG.OUTPUT_DIR, CFG.MODELS_DIR, CFG.LOGS_DIR, CFG.FIGURES_DIR, CFG.OPTUNA_DIR,
          CFG.SUBMISSIONS_DIR, CFG.ATTENTION_DIR, CFG.CACHE_DIR]:
    logger.info(f"  -> {d.resolve()}")


def seed_everything(seed: int = CFG.SEED) -> None:
    '''Seed every RNG we touch, including CUDA determinism knobs.'''
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    logger.info(f"Global seed set to {seed}")


seed_everything(CFG.SEED)


## 5. Data Loading, Memory Optimization, Missing-Value Handling

In [ ]:

def reduce_mem_usage(df: pd.DataFrame, name: str = "df", verbose: bool = True) -> pd.DataFrame:
    start_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    for col in tqdm(df.columns, desc=f"Reducing memory [{name}]", leave=False):
        col_type = df[col].dtype
        if col_type == object or str(col_type) == "category":
            continue
        c_min, c_max = df[col].min(), df[col].max()
        if pd.api.types.is_integer_dtype(col_type):
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    if verbose:
        logger.info(f"[{name}] Memory: {start_mem:,.1f} MB -> {end_mem:,.1f} MB "
                    f"({100 * (start_mem - end_mem) / max(start_mem, 1e-9):.1f}% reduction)")
    return df


def load_csv_optimized(path: Path, name: str, dtype_map: Optional[dict] = None) -> pd.DataFrame:
    logger.info(f"Loading [{name}] from {path} ...")
    t0 = time.time()
    if not path.exists():
        raise FileNotFoundError(f"Expected data file not found: {path}")
    df = pd.read_csv(path, dtype=dtype_map, low_memory=False)
    logger.info(f"[{name}] loaded: shape={df.shape}, time={time.time() - t0:.1f}s")
    if CFG.REDUCE_MEMORY:
        df = reduce_mem_usage(df, name=name)
    return df


HORIZONTAL_DTYPES = {
    "MD": "float32", "X": "float32", "Y": "float32", "Z": "float32",
    "ANCC": "float32", "ASTNU": "float32", "ASTNL": "float32",
    "EGFDU": "float32", "EGFDL": "float32", "BUDA": "float32",
    "TVT": "float32", "GR": "float32", "TVT_input": "float32",
    "WELL_NAME": "object",
}
TYPEWELL_DTYPES = {"TVT": "float32", "GR": "float32", "Geology": "object", "WELL_NAME": "object"}


def load_all_data(cfg: Config = CFG):
    train_path = cfg.DATA_DIR / cfg.TRAIN_FILE
    typewell_path = cfg.DATA_DIR / cfg.TYPEWELL_FILE
    test_path = cfg.DATA_DIR / cfg.TEST_FILE

    train_df = load_csv_optimized(train_path, "train", dtype_map=HORIZONTAL_DTYPES)
    typewell_df = load_csv_optimized(typewell_path, "typewell", dtype_map=TYPEWELL_DTYPES)

    test_df = None
    if test_path.exists():
        test_dtypes = {k: v for k, v in HORIZONTAL_DTYPES.items() if k != "TVT"}
        test_df = load_csv_optimized(test_path, "test", dtype_map=test_dtypes)
    else:
        logger.warning(f"Test file not found at {test_path} — inference cell will need it later.")

    if "TVT" in train_df.columns:
        train_df = train_df.rename(columns={"TVT": cfg.TARGET_COL})

    logger.info(f"train shape   : {train_df.shape}")
    logger.info(f"typewell shape: {typewell_df.shape}")
    if test_df is not None:
        logger.info(f"test shape    : {test_df.shape}")
    logger.info(f"Unique wells in train: {train_df[cfg.GROUP_COL].nunique()}")
    return train_df, typewell_df, test_df


train_df, typewell_df, test_df = load_all_data(CFG)


def handle_missing_values(df: pd.DataFrame, cols: List[str], group_col: str = CFG.GROUP_COL,
                           md_col: str = CFG.MD_COL) -> pd.DataFrame:
    df = df.sort_values([group_col, md_col]).reset_index(drop=True)
    existing_cols = [c for c in cols if c in df.columns]
    n_before = df[existing_cols].isna().sum().sum()
    logger.info(f"Missing values before imputation: {n_before:,}")
    for col in tqdm(existing_cols, desc="Imputing missing values"):
        if df[col].isna().sum() == 0:
            continue
        df[col] = df.groupby(group_col, sort=False, observed=True)[col].transform(
            lambda s: s.interpolate(method="linear", limit_direction="both"))
        still_na = df[col].isna()
        if still_na.any():
            df[col] = df.groupby(group_col, sort=False, observed=True)[col].transform(lambda s: s.ffill().bfill())
        still_na = df[col].isna()
        if still_na.any():
            well_median = df.groupby(group_col, observed=True)[col].transform("median")
            df.loc[still_na, col] = well_median[still_na]
        still_na = df[col].isna()
        if still_na.any():
            df.loc[still_na, col] = df[col].median()
    n_after = df[existing_cols].isna().sum().sum()
    logger.info(f"Missing values after imputation: {n_after:,}")
    assert n_after == 0
    return df


numeric_feature_cols = list(CFG.HORIZONTAL_LOG_COLS) + list(CFG.COORD_COLS) + ["TVT_input", CFG.MD_COL]
train_df = handle_missing_values(train_df, numeric_feature_cols)
if test_df is not None:
    test_df = handle_missing_values(test_df, numeric_feature_cols)

typewell_df = typewell_df.sort_values([CFG.GROUP_COL, CFG.TYPEWELL_TVT_COL]).reset_index(drop=True)
typewell_df[CFG.TYPEWELL_GR_COL] = typewell_df.groupby(CFG.GROUP_COL, observed=True)[CFG.TYPEWELL_GR_COL] \
    .transform(lambda s: s.interpolate(method="linear", limit_direction="both").ffill().bfill())
typewell_df[CFG.TYPEWELL_GEOLOGY_COL] = typewell_df.groupby(CFG.GROUP_COL, observed=True)[CFG.TYPEWELL_GEOLOGY_COL] \
    .transform(lambda s: s.ffill().bfill())


## 6. Typewell Attachment & Geology Encoding

In [ ]:

def attach_typewell_geology(df: pd.DataFrame, typewell_df: pd.DataFrame,
                             on_col: str = "TVT_input") -> pd.DataFrame:
    df = df.sort_values([CFG.GROUP_COL, on_col]).reset_index(drop=True)
    tw = typewell_df.sort_values([CFG.GROUP_COL, CFG.TYPEWELL_TVT_COL]).reset_index(drop=True)

    merged_chunks = []
    for well, sub in tqdm(df.groupby(CFG.GROUP_COL, sort=False, observed=True),
                           desc="Attaching typewell geology (merge_asof per well)"):
        tw_sub = tw[tw[CFG.GROUP_COL] == well]
        if tw_sub.empty:
            sub = sub.copy()
            sub["Geology"] = "UNKNOWN"
            sub["typewell_GR_ref"] = np.nan
            sub["typewell_TVT_ref"] = np.nan
            merged_chunks.append(sub)
            continue
        merged = pd.merge_asof(
            sub.sort_values(on_col),
            tw_sub[[CFG.TYPEWELL_TVT_COL, CFG.TYPEWELL_GR_COL, CFG.TYPEWELL_GEOLOGY_COL]]
                .rename(columns={CFG.TYPEWELL_GR_COL: "typewell_GR_ref",
                                  CFG.TYPEWELL_TVT_COL: "typewell_TVT_ref",
                                  CFG.TYPEWELL_GEOLOGY_COL: "Geology"})
                .sort_values("typewell_TVT_ref"),
            left_on=on_col, right_on="typewell_TVT_ref", direction="nearest",
        )
        merged_chunks.append(merged)

    out = pd.concat(merged_chunks, ignore_index=True)
    out["Geology"] = out["Geology"].fillna("UNKNOWN")
    return out


train_df = attach_typewell_geology(train_df, typewell_df)
if test_df is not None:
    test_df = attach_typewell_geology(test_df, typewell_df)


geology_encoder = LabelEncoder()
train_geo_vals = train_df["Geology"].astype(str).fillna("UNKNOWN")
geology_encoder.fit(list(train_geo_vals.unique()) + ["UNKNOWN"])
train_df["Geology_idx"] = geology_encoder.transform(train_geo_vals).astype(np.int64)
N_GEOLOGY_CLASSES = len(geology_encoder.classes_)

if test_df is not None:
    test_geo_vals = test_df["Geology"].astype(str).fillna("UNKNOWN")
    test_geo_vals = test_geo_vals.where(test_geo_vals.isin(geology_encoder.classes_), "UNKNOWN")
    test_df["Geology_idx"] = geology_encoder.transform(test_geo_vals).astype(np.int64)

with open(CFG.CACHE_DIR / "geology_encoder.pkl", "wb") as f:
    pickle.dump(geology_encoder, f)

logger.info(f"Geology classes: {N_GEOLOGY_CLASSES} -> {list(geology_encoder.classes_)}")


## 7. Feature Engineering (Lean, Sequence-Model-Appropriate)

The CNN + Transformer stack learns local and long-range temporal patterns directly from
the raw sequence, so — as with the other sequence models in this project — we skip the
rolling/lag/polynomial feature explosion used in the tree-based notebooks and keep only
positional, distance, and geological-offset features the network can't trivially derive
on its own.

In [ ]:
def engineer_sequence_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values([CFG.GROUP_COL, CFG.MD_COL]).reset_index(drop=True)
    g = df.groupby(CFG.GROUP_COL, observed=True)

    for c in CFG.COORD_COLS:
        if c in df.columns:
            df[f"{c}_rel_start"] = df[c] - g[c].transform("first")

    if all(c in df.columns for c in ("X", "Y", "Z")):
        dx = g["X"].diff().fillna(0.0)
        dy = g["Y"].diff().fillna(0.0)
        dz = g["Z"].diff().fillna(0.0)
        step_dist = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2).astype(np.float32)
        df["step_distance_3d"] = step_dist
        df["cum_distance_3d"] = df.groupby(CFG.GROUP_COL, observed=True)["step_distance_3d"].cumsum().astype(np.float32)

        step_horiz = np.sqrt(dx ** 2 + dy ** 2).astype(np.float32)
        df["step_distance_horizontal"] = step_horiz
        df["cum_distance_horizontal"] = df.groupby(CFG.GROUP_COL, observed=True)["step_distance_horizontal"].cumsum().astype(np.float32)

        first_x, first_y, first_z = g["X"].transform("first"), g["Y"].transform("first"), g["Z"].transform("first")
        df["dist_from_heel"] = np.sqrt((df["X"] - first_x) ** 2 + (df["Y"] - first_y) ** 2 + (df["Z"] - first_z) ** 2).astype(np.float32)

        dmd = g[CFG.MD_COL].diff().replace(0, np.nan)
        df["inclination_proxy"] = np.degrees(np.arctan2(step_horiz, dz.abs() + 1e-6)).astype(np.float32)
        df["tortuosity_proxy"] = (step_dist / (dmd.abs() + 1e-6)).clip(0, 10).fillna(1.0).astype(np.float32)
    else:
        dmd = g[CFG.MD_COL].diff().replace(0, np.nan)

    md_g = df.groupby(CFG.GROUP_COL, observed=True)[CFG.MD_COL]
    df["md_frac_in_well"] = ((df[CFG.MD_COL] - md_g.transform("min")) /
                              (md_g.transform("max") - md_g.transform("min") + 1e-6)).astype(np.float32)

    if "typewell_TVT_ref" in df.columns and "TVT_input" in df.columns:
        df["tvt_geo_offset"] = (df["TVT_input"] - df["typewell_TVT_ref"]).astype(np.float32)
    if "typewell_GR_ref" in df.columns and "GR" in df.columns:
        df["gr_vs_typewell_diff"] = (df["GR"] - df["typewell_GR_ref"]).astype(np.float32)

    # NOTE: a test set may not include every log curve train has (e.g. only GR is
    # measured while drilling in real time, while resistivity/density curves are only
    # available on the typewell/pilot). Every per-column feature below is therefore
    # guarded so a missing raw column is skipped rather than raising a KeyError.
    for col in list(CFG.HORIZONTAL_LOG_COLS) + ["TVT_input"]:
        if col not in df.columns:
            continue
        d1 = df.groupby(CFG.GROUP_COL, observed=True)[col].diff()
        df[f"{col}_grad1"] = (d1 / dmd).replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)

    df = df.replace([np.inf, -np.inf], np.nan)
    float_cols = df.select_dtypes(include=[np.floating]).columns
    df[float_cols] = df[float_cols].fillna(0.0)
    return df


_full_candidate_cols = (
    list(CFG.HORIZONTAL_LOG_COLS) + ["TVT_input", "md_frac_in_well"]
    + [f"{c}_rel_start" for c in CFG.COORD_COLS]
    + ["cum_distance_3d", "cum_distance_horizontal", "dist_from_heel",
       "inclination_proxy", "tortuosity_proxy"]
    + ["tvt_geo_offset", "gr_vs_typewell_diff", "typewell_GR_ref", "typewell_TVT_ref"]
    + [f"{c}_grad1" for c in list(CFG.HORIZONTAL_LOG_COLS) + ["TVT_input"]]
)

t0 = time.time()
train_df = engineer_sequence_features(train_df)
logger.info(f"Sequence feature engineering (train) done in {time.time()-t0:.1f}s -> shape {train_df.shape}")
if test_df is not None:
    t0 = time.time()
    test_df = engineer_sequence_features(test_df)
    logger.info(f"Sequence feature engineering (test) done in {time.time()-t0:.1f}s -> shape {test_df.shape}")

# Only keep feature channels present in train AND (when available) test, so the model's
# input dimensionality stays consistent and inference never crashes on a column the test
# set doesn't provide.
CONTINUOUS_FEATURE_COLS = [c for c in _full_candidate_cols if c in train_df.columns]
if test_df is not None:
    missing_in_test = [c for c in CONTINUOUS_FEATURE_COLS if c not in test_df.columns]
    if missing_in_test:
        logger.warning(f"Dropping {len(missing_in_test)} feature column(s) not present in test set: {missing_in_test}")
    CONTINUOUS_FEATURE_COLS = [c for c in CONTINUOUS_FEATURE_COLS if c not in missing_in_test]

logger.info(f"Continuous feature channels ({len(CONTINUOUS_FEATURE_COLS)}): {CONTINUOUS_FEATURE_COLS}")
gc.collect()

## 8. GroupKFold Cross-Validation (split on `WELL_NAME`)

In [ ]:

def make_group_folds(df: pd.DataFrame, n_folds: int = CFG.N_FOLDS,
                      group_col: str = CFG.GROUP_COL, seed: int = CFG.SEED) -> pd.DataFrame:
    gkf = GroupKFold(n_splits=n_folds)
    df = df.copy()
    df["fold"] = -1
    groups = df[group_col].astype(str).values
    unique_wells = df[group_col].astype(str).unique()
    rng = np.random.default_rng(seed)
    shuffled_wells = rng.permutation(unique_wells)
    well_to_rank = {w: i for i, w in enumerate(shuffled_wells)}
    order = df[group_col].astype(str).map(well_to_rank).sort_values().index

    for fold, (_, val_idx) in enumerate(gkf.split(df.loc[order], groups=groups[order])):
        df.loc[order[val_idx], "fold"] = fold

    fold_summary = df.groupby("fold").agg(n_rows=(group_col, "size"), n_wells=(group_col, "nunique"))
    logger.info(f"GroupKFold summary:\n{fold_summary.to_string()}")

    for f1 in range(n_folds):
        wells_f1 = set(df.loc[df.fold == f1, group_col].astype(str))
        for f2 in range(f1 + 1, n_folds):
            wells_f2 = set(df.loc[df.fold == f2, group_col].astype(str))
            assert len(wells_f1 & wells_f2) == 0, f"Well leakage between fold {f1} and {f2}"
    logger.info("Verified: zero well overlap across folds.")
    return df


train_df = make_group_folds(train_df, n_folds=CFG.N_FOLDS)


## 9. Fold-Safe Feature & Target Scaling

**Fix applied here:** in addition to standardizing the continuous feature channels
(as before), `Target_TVT` is now also standardized per fold (fit on training rows
only). Previously the raw ~10,000-ft target was regressed directly, which produced
a badly-conditioned loss (`train_mse` in the hundreds of millions at epoch 1),
overflowed fp16 under `autocast` mid-training (the `loss=nan` epochs), and made
each fold's absolute-value regression an extrapolation problem across wells with
different TVT ranges — which is what caused 2 of 5 folds to blow up to
RMSE ~960-1600 while the other 3 sat at a reasonable ~22-24.

In [ ]:

def fit_scaler_on_fold(train_fold_df: pd.DataFrame, cols: List[str]) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(train_fold_df[cols].values)
    return scaler


def scale_df(df: pd.DataFrame, cols: List[str], scaler: StandardScaler) -> pd.DataFrame:
    df = df.copy()
    df[cols] = scaler.transform(df[cols].values).astype(np.float32)
    return df


# ----------------------------------------------------------------------------
# Target scaling (FIX: previously Target_TVT was regressed in raw ~10,000-ft
# units with plain MSE. That gives a huge, badly-conditioned loss surface,
# overflows fp16 under autocast (squared errors of a few hundred feet already
# exceed fp16's ~65,504 max -> the `train_mse=nan` seen mid-training), and
# forces every fold's absolute-value regression to extrapolate across wells
# with very different TVT ranges. Standardizing the target per fold (fit on
# the training rows only, to avoid leakage) fixes all three at once.
# ----------------------------------------------------------------------------
def fit_target_scaler(train_fold_df: pd.DataFrame, target_col: str) -> Dict[str, float]:
    mean = float(train_fold_df[target_col].mean())
    std = float(train_fold_df[target_col].std())
    if not np.isfinite(std) or std < 1e-6:
        std = 1.0
    return {"mean": mean, "std": std}


def scale_target_col(df: pd.DataFrame, target_col: str, target_stats: Dict[str, float],
                      out_col: str = "_y_scaled") -> pd.DataFrame:
    df = df.copy()
    df[out_col] = ((df[target_col] - target_stats["mean"]) / target_stats["std"]).astype(np.float32)
    return df


def inverse_transform_target(y_scaled: np.ndarray, target_stats: Dict[str, float]) -> np.ndarray:
    return y_scaled * target_stats["std"] + target_stats["mean"]


SCALED_TARGET_COL = "_y_scaled"

logger.info("Scaling helpers ready (features + target).")


## 10. Sliding-Window Dataset

Each well's MD-ordered sequence is cut into fixed-length overlapping windows. Unlike the
TCN+CrossAttention+Transformer notebook, there is no separate typewell-context tensor
here — the geological-offset signal already lives in the per-row continuous features.

In [ ]:

class WellWindowDataset(Dataset):
    '''Sliding-window dataset over MD-ordered per-well sequences. No typewell context
    tensor here (unlike the cross-attention model) — geological-offset information is
    already present as scalar per-row features in `continuous_cols`.'''

    def __init__(self, df: pd.DataFrame, continuous_cols: List[str], window_size: int,
                 stride: int, has_target: bool = True, cfg: Config = CFG,
                 target_col: Optional[str] = None):
        self.cfg = cfg
        self.continuous_cols = continuous_cols
        self.window_size = window_size
        self.has_target = has_target
        # target_col lets callers point this at a standardized target column
        # (e.g. SCALED_TARGET_COL) instead of always reading raw cfg.TARGET_COL.
        self.target_col = target_col or cfg.TARGET_COL
        self.samples = []

        self.well_arrays: Dict[str, dict] = {}
        for well, sub in df.groupby(cfg.GROUP_COL, observed=True):
            sub = sub.sort_values(cfg.MD_COL)
            arr = {
                "X": sub[continuous_cols].values.astype(np.float32),
                "geo": sub["Geology_idx"].values.astype(np.int64),
            }
            if has_target:
                arr["y"] = sub[self.target_col].values.astype(np.float32)
            arr["row_index"] = sub.index.values
            self.well_arrays[well] = arr

            n = len(sub)
            if n <= window_size:
                self.samples.append((well, 0, n))
            else:
                start = 0
                while start < n:
                    end = min(start + window_size, n)
                    self.samples.append((well, start, end))
                    if end == n:
                        break
                    start += stride

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        well, start, end = self.samples[idx]
        arr = self.well_arrays[well]
        n_valid = end - start
        W = self.window_size

        x = np.zeros((W, len(self.continuous_cols)), dtype=np.float32)
        x[:n_valid] = arr["X"][start:end]
        geo = np.zeros(W, dtype=np.int64)
        geo[:n_valid] = arr["geo"][start:end]
        mask = np.zeros(W, dtype=np.float32)
        mask[:n_valid] = 1.0
        row_idx = -np.ones(W, dtype=np.int64)
        row_idx[:n_valid] = arr["row_index"][start:end]

        item = {
            "x": torch.from_numpy(x), "geo": torch.from_numpy(geo),
            "mask": torch.from_numpy(mask), "row_idx": torch.from_numpy(row_idx),
        }
        if self.has_target:
            y = np.zeros(W, dtype=np.float32)
            y[:n_valid] = arr["y"][start:end]
            item["y"] = torch.from_numpy(y)
        return item


def collate_windows(batch: List[dict]) -> dict:
    return {key: torch.stack([b[key] for b in batch], dim=0) for key in batch[0].keys()}


logger.info("WellWindowDataset defined.")


## 11. Model — CNN + Transformer

- `ConvBlock` / `CNNEncoder`: stacked `Conv1d` + `BatchNorm1d` + `ReLU` + `Dropout` with a
  1×1 residual projection when channel counts change — a standard (non-dilated) local
  feature extractor.
- `PositionalEncoding`: sinusoidal, added before the Transformer stage.
- `SelfAttentionLayer` / `TransformerStack`: a hand-rolled Transformer encoder layer built
  directly on `nn.MultiheadAttention` (rather than `nn.TransformerEncoderLayer`) so that
  self-attention weights can be returned for visualization.
- Output head: a small per-timestep MLP regression head.

In [ ]:

class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, dropout: float):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm1d(out_channels)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None

    def forward(self, x):
        out = self.dropout(self.act(self.bn(self.conv(x))))
        res = x if self.downsample is None else self.downsample(x)
        return out + res


class CNNEncoder(nn.Module):
    def __init__(self, in_channels: int, channels: Tuple[int, ...], kernel_size: int, dropout: float):
        super().__init__()
        layers = []
        prev_c = in_channels
        for c in channels:
            layers.append(ConvBlock(prev_c, c, kernel_size, dropout))
            prev_c = c
        self.net = nn.Sequential(*layers)
        self.out_channels = prev_c

    def forward(self, x):  # x: [B, C_in, T]
        return self.net(x)  # [B, C_out, T]


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class SelfAttentionLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ff_dim: int, dropout: float):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, ff_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_dim, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        attn_out, attn_weights = self.attn(x, x, x, key_padding_mask=key_padding_mask,
                                            need_weights=True, average_attn_weights=True)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x, attn_weights


class TransformerStack(nn.Module):
    def __init__(self, d_model: int, n_layers: int, n_heads: int, ff_dim: int, dropout: float):
        super().__init__()
        self.layers = nn.ModuleList([SelfAttentionLayer(d_model, n_heads, ff_dim, dropout) for _ in range(n_layers)])

    def forward(self, x, key_padding_mask=None):
        attn_weights = None
        for layer in self.layers:
            x, attn_weights = layer(x, key_padding_mask)
        return x, attn_weights  # returns the LAST layer's self-attention weights for viz


class SequenceModel(nn.Module):
    def __init__(self, n_continuous_features: int, geology_vocab: int, cfg: Config = CFG):
        super().__init__()
        self.cfg = cfg
        self.geo_embed = nn.Embedding(geology_vocab, cfg.GEOLOGY_EMBED_DIM)

        cnn_in_channels = n_continuous_features + cfg.GEOLOGY_EMBED_DIM
        self.cnn = CNNEncoder(cnn_in_channels, cfg.CNN_CHANNELS, cfg.CNN_KERNEL_SIZE, cfg.CNN_DROPOUT)

        self.input_proj = nn.Linear(self.cnn.out_channels, cfg.D_MODEL) if self.cnn.out_channels != cfg.D_MODEL else nn.Identity()
        self.pos_enc = PositionalEncoding(cfg.D_MODEL)
        self.transformer = TransformerStack(cfg.D_MODEL, cfg.N_TRANSFORMER_LAYERS, cfg.N_TRANSFORMER_HEADS,
                                             cfg.TRANSFORMER_FF_DIM, cfg.TRANSFORMER_DROPOUT)

        self.head = nn.Sequential(
            nn.Linear(cfg.D_MODEL, cfg.HEAD_HIDDEN_DIM), nn.GELU(), nn.Dropout(cfg.HEAD_DROPOUT),
            nn.Linear(cfg.HEAD_HIDDEN_DIM, 1),
        )

    def forward(self, x_cont, geo_idx, mask, return_attention: bool = False):
        geo_emb = self.geo_embed(geo_idx)
        x = torch.cat([x_cont, geo_emb], dim=-1).transpose(1, 2)  # [B, F+E, T]

        h = self.cnn(x).transpose(1, 2)  # [B, T, C_cnn]
        h = self.input_proj(h)
        h = self.pos_enc(h)

        key_padding_mask = (mask == 0)
        h, attn_weights = self.transformer(h, key_padding_mask)

        pred = self.head(h).squeeze(-1)
        if return_attention:
            return pred, attn_weights
        return pred


def build_model(n_continuous_features: int, geology_vocab: int, cfg: Config = CFG) -> nn.Module:
    model = SequenceModel(n_continuous_features, geology_vocab, cfg).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Model built: {n_params:,} trainable parameters")
    return model


logger.info("Model architecture defined.")


## 12. Masked Loss & Metrics

In [ ]:

def masked_mse_loss(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    sq_err = (pred - target) ** 2 * mask
    return sq_err.sum() / mask.sum().clamp(min=1.0)


def masked_rmse(pred: np.ndarray, target: np.ndarray, mask: np.ndarray) -> float:
    sq_err = ((pred - target) ** 2) * mask
    return float(np.sqrt(sq_err.sum() / max(mask.sum(), 1.0)))


logger.info("Loss/metric functions defined.")


## 13. Training Loop (AMP, Gradient Clipping, OneCycle LR, Early Stopping)

In [ ]:

def run_epoch(model: nn.Module, loader: DataLoader, optimizer=None, scaler: Optional[GradScaler] = None,
              scheduler=None, grad_clip: float = 1.0, train: bool = True) -> float:
    model.train(mode=train)
    total_loss, total_count = 0.0, 0

    with torch.set_grad_enabled(train):
        for batch in loader:
            x = batch["x"].to(DEVICE, non_blocking=True)
            geo = batch["geo"].to(DEVICE, non_blocking=True)
            mask = batch["mask"].to(DEVICE, non_blocking=True)
            y = batch["y"].to(DEVICE, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=CFG.USE_AMP):
                pred = model(x, geo, mask)
            # FIX: compute the loss itself in fp32, outside autocast. The forward
            # pass can still run in fp16 for speed, but squaring the raw error in
            # fp16 is what produced the `loss=nan` epochs — fp16 tops out at
            # ~65,504 and a single moderately-off prediction squares past that.
            loss = masked_mse_loss(pred.float(), y.float(), mask)

            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()

            n = mask.sum().item()
            total_loss += loss.item() * n
            total_count += n

    return total_loss / max(total_count, 1)


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                 max_epochs: int, learning_rate: float, weight_decay: float,
                 grad_clip: float, patience: int, trial: Optional[optuna.Trial] = None,
                 fold: Optional[int] = None, target_std: float = 1.0) -> Tuple[nn.Module, dict]:
    """target_std: the training-fold target's standard deviation (feet). Loss/RMSE are
    computed on the standardized target internally, but history/logging/early-stopping
    below are converted back to real feet (RMSE_real = RMSE_scaled * target_std, since
    the mean offset cancels out of an error term) so numbers are directly comparable
    across folds and against the other notebook's RMSE."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=learning_rate, total_steps=max_epochs * max(len(train_loader), 1),
        pct_start=0.1, anneal_strategy="cos",
    )
    scaler = GradScaler(enabled=CFG.USE_AMP)

    best_val_rmse = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_rmse": []}

    for epoch in range(max_epochs):
        t0 = time.time()
        train_loss_scaled = run_epoch(model, train_loader, optimizer, scaler, scheduler, grad_clip, train=True)
        val_loss_scaled = run_epoch(model, val_loader, train=False)
        train_loss = train_loss_scaled * (target_std ** 2)   # real-unit MSE (ft^2)
        val_loss = val_loss_scaled * (target_std ** 2)
        val_rmse = math.sqrt(val_loss_scaled) * target_std   # real-unit RMSE (ft)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_rmse"].append(val_rmse)

        logger.info(f"[fold={fold}] epoch {epoch+1}/{max_epochs} | "
                    f"train_mse={train_loss:.5f} val_mse={val_loss:.5f} val_rmse={val_rmse:.5f} "
                    f"({time.time()-t0:.1f}s)")

        if trial is not None:
            trial.report(val_rmse, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if val_rmse < best_val_rmse - 1e-5:
            best_val_rmse = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                logger.info(f"[fold={fold}] Early stopping at epoch {epoch+1} (best val_rmse={best_val_rmse:.5f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, {"best_val_rmse": best_val_rmse, "history": history}


logger.info("Training loop defined.")


## 14. Optuna Hyperparameter Search

Search covers CNN depth/width/kernel size, Transformer depth/width/heads, embedding
dimension, window size, dropout rates, learning rate, weight decay, and batch size.
Pruning is fold-level (`trial.report`/`should_prune` inside the training loop).

In [ ]:

def build_model_hparams(trial: optuna.Trial) -> dict:
    n_cnn_layers = trial.suggest_int("n_cnn_layers", 2, 8)
    cnn_base_channels = trial.suggest_categorical("cnn_base_channels", [32, 48, 64, 96])
    cnn_channels = tuple(cnn_base_channels * (2 ** min(i, 2)) for i in range(n_cnn_layers))

    d_model = trial.suggest_categorical("d_model", [64, 96, 128, 192, 256])
    n_heads_candidates = [h for h in [2, 4, 8, 16] if d_model % h == 0]

    return {
        "CNN_CHANNELS": cnn_channels,
        "CNN_KERNEL_SIZE": trial.suggest_categorical("cnn_kernel_size", [3, 5, 7, 9]),
        "CNN_DROPOUT": trial.suggest_float("cnn_dropout", 0.0, 0.4),
        "GEOLOGY_EMBED_DIM": trial.suggest_categorical("geology_embed_dim", [4, 8, 16, 32]),
        "D_MODEL": d_model,
        "N_TRANSFORMER_LAYERS": trial.suggest_int("n_transformer_layers", 1, 6),
        "N_TRANSFORMER_HEADS": trial.suggest_categorical("n_transformer_heads", n_heads_candidates),
        "TRANSFORMER_FF_DIM": trial.suggest_categorical("transformer_ff_dim", [128, 256, 512, 1024]),
        "TRANSFORMER_DROPOUT": trial.suggest_float("transformer_dropout", 0.0, 0.4),
        "HEAD_HIDDEN_DIM": trial.suggest_categorical("head_hidden_dim", [64, 128, 256]),
        "HEAD_DROPOUT": trial.suggest_float("head_dropout", 0.0, 0.4),
    }


def build_train_hparams(trial: optuna.Trial) -> dict:
    return {
        "LEARNING_RATE": trial.suggest_float("learning_rate", 1e-5, 5e-3, log=True),
        "WEIGHT_DECAY": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "BATCH_SIZE": trial.suggest_categorical("batch_size", [64, 128, 256, 512]),
        "GRAD_CLIP_NORM": trial.suggest_float("grad_clip_norm", 0.1, 5.0, log=True),
        "WINDOW_SIZE": trial.suggest_categorical("window_size", [128, 256, 512]),
    }


In [ ]:

def make_fold_config(base_cfg: Config, model_hp: dict, train_hp: dict) -> Config:
    cfg_dict = asdict(base_cfg)
    cfg_dict.update(model_hp)
    cfg_dict.update(train_hp)
    # asdict() turns Path objects into strings and tuples into lists — restore types
    for path_field in ["DATA_DIR", "OUTPUT_DIR", "MODELS_DIR", "LOGS_DIR", "FIGURES_DIR",
                        "OPTUNA_DIR", "SUBMISSIONS_DIR", "ATTENTION_DIR", "CACHE_DIR"]:
        cfg_dict[path_field] = Path(cfg_dict[path_field])
    for tuple_field in ["CNN_CHANNELS"] + ["HORIZONTAL_LOG_COLS", "COORD_COLS"]:
        if tuple_field in cfg_dict:
            cfg_dict[tuple_field] = tuple(cfg_dict[tuple_field])
    return Config(**cfg_dict)


In [ ]:

def optuna_objective(trial: optuna.Trial, folds_df: pd.DataFrame, n_search_folds: int) -> float:
    model_hp = build_model_hparams(trial)
    train_hp = build_train_hparams(trial)
    fold_cfg = make_fold_config(CFG, model_hp, train_hp)

    fold_rmses = []
    search_folds = list(range(min(n_search_folds, CFG.N_FOLDS)))

    for fold in search_folds:
        tr = folds_df[folds_df.fold != fold].copy()
        va = folds_df[folds_df.fold == fold].copy()

        scaler = fit_scaler_on_fold(tr, CONTINUOUS_FEATURE_COLS)
        tr = scale_df(tr, CONTINUOUS_FEATURE_COLS, scaler)
        va = scale_df(va, CONTINUOUS_FEATURE_COLS, scaler)

        target_stats = fit_target_scaler(tr, CFG.TARGET_COL)
        tr = scale_target_col(tr, CFG.TARGET_COL, target_stats)
        va = scale_target_col(va, CFG.TARGET_COL, target_stats)

        train_ds = WellWindowDataset(tr, CONTINUOUS_FEATURE_COLS, fold_cfg.WINDOW_SIZE,
                                      fold_cfg.WINDOW_STRIDE_TRAIN, has_target=True, cfg=fold_cfg,
                                      target_col=SCALED_TARGET_COL)
        val_ds = WellWindowDataset(va, CONTINUOUS_FEATURE_COLS, fold_cfg.WINDOW_SIZE,
                                    fold_cfg.WINDOW_STRIDE_TRAIN, has_target=True, cfg=fold_cfg,
                                    target_col=SCALED_TARGET_COL)

        train_loader = DataLoader(train_ds, batch_size=fold_cfg.BATCH_SIZE, shuffle=True,
                                   num_workers=min(8, CFG.NUM_WORKERS), collate_fn=collate_windows,
                                   pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=fold_cfg.BATCH_SIZE, shuffle=False,
                                 num_workers=min(8, CFG.NUM_WORKERS), collate_fn=collate_windows,
                                 pin_memory=True)

        model = build_model(len(CONTINUOUS_FEATURE_COLS), N_GEOLOGY_CLASSES, fold_cfg)
        try:
            _, result = train_model(
                model, train_loader, val_loader,
                max_epochs=CFG.OPTUNA_MAX_EPOCHS, learning_rate=fold_cfg.LEARNING_RATE,
                weight_decay=fold_cfg.WEIGHT_DECAY, grad_clip=fold_cfg.GRAD_CLIP_NORM,
                patience=CFG.OPTUNA_EARLY_STOPPING_PATIENCE, trial=trial, fold=fold,
                target_std=target_stats["std"],
            )
        finally:
            del model
            torch.cuda.empty_cache()
            gc.collect()

        fold_rmses.append(result["best_val_rmse"])

    return float(np.mean(fold_rmses))


logger.info("Optuna objective function defined.")


## 15. Run Optuna Study (SQLite-Backed, Resumable, Pruned, Checkpointed)

In [ ]:

def get_or_create_study(cfg: Config = CFG) -> optuna.Study:
    sampler = TPESampler(seed=cfg.SEED, n_startup_trials=cfg.OPTUNA_N_STARTUP_TRIALS, multivariate=True)
    pruner = MedianPruner(n_startup_trials=max(3, cfg.OPTUNA_N_STARTUP_TRIALS // 3),
                           n_warmup_steps=3, interval_steps=1)
    study = optuna.create_study(
        study_name=cfg.OPTUNA_STUDY_NAME, storage=cfg.OPTUNA_STORAGE,
        direction="minimize", sampler=sampler, pruner=pruner, load_if_exists=True,
    )
    n_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    logger.info(f"Loaded/created study '{cfg.OPTUNA_STUDY_NAME}' "
                f"({n_done} complete, {n_pruned} pruned, {len(study.trials)} total trials so far)")
    return study


def safe_best(study: optuna.Study) -> Tuple[Optional[float], Optional[dict]]:
    try:
        return study.best_value, study.best_params
    except ValueError:
        return None, None


def checkpoint_callback(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
    if trial.number % CFG.OPTUNA_CHECKPOINT_EVERY == 0:
        study.trials_dataframe().to_csv(CFG.OPTUNA_DIR / "trials_checkpoint.csv", index=False)
        best_value, best_params = safe_best(study)
        if best_value is not None:
            with open(CFG.OPTUNA_DIR / "best_params_running.json", "w") as f:
                json.dump({"best_value": best_value, "best_params": best_params,
                           "n_trials_so_far": len(study.trials)}, f, indent=2)
        best_str = f"{best_value:.6f}" if best_value is not None else "n/a"
        logger.info(f"[checkpoint] trial={trial.number} value={trial.value} best_so_far={best_str}")


RUN_OPTUNA_SEARCH = True   # flip to False to skip straight to a saved best_params.json

if RUN_OPTUNA_SEARCH:
    study = get_or_create_study(CFG)
    n_already_done = len([t for t in study.trials if t.state != optuna.trial.TrialState.RUNNING])
    n_remaining = max(0, CFG.N_OPTUNA_TRIALS - n_already_done)
    logger.info(f"Trials already recorded: {n_already_done} | remaining to reach target: {n_remaining}")

    if n_remaining > 0:
        objective_fn = lambda trial: optuna_objective(trial, train_df, CFG.OPTUNA_CV_FOLDS_FOR_SEARCH)
        with tqdm(total=n_remaining, desc="Optuna trials") as pbar:
            def _tqdm_cb(study, trial):
                pbar.update(1)
                checkpoint_callback(study, trial)

            study.optimize(
                objective_fn, n_trials=n_remaining, timeout=CFG.OPTUNA_TIMEOUT,
                n_jobs=1, gc_after_trial=True, callbacks=[_tqdm_cb],
                catch=(RuntimeError, torch.cuda.OutOfMemoryError),
            )

    best_value, best_params = safe_best(study)
    if best_value is not None:
        logger.info(f"Optuna search complete. Best RMSE: {best_value:.6f}")
        logger.info(f"Best params: {json.dumps(best_params, indent=2)}")
        with open(CFG.OPTUNA_DIR / "best_params.json", "w") as f:
            json.dump({"best_value": best_value, "best_params": best_params}, f, indent=2)
    else:
        logger.warning("No trial completed successfully — falling back to hand-tuned defaults.")

    study.trials_dataframe().to_csv(CFG.OPTUNA_DIR / "all_trials.csv", index=False)
else:
    logger.info("Skipping Optuna search — will load best_params.json if present.")
    study = None


## 16. Optuna Visualization

In [ ]:

def save_optuna_visualizations(study: optuna.Study, out_dir: Path = CFG.OPTUNA_DIR) -> None:
    if study is None or len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]) < 2:
        logger.warning("Not enough completed trials to visualize — skipping.")
        return
    figs = {
        "optimization_history": opt_viz.plot_optimization_history(study),
        "param_importances": opt_viz.plot_param_importances(study),
        "parallel_coordinate": opt_viz.plot_parallel_coordinate(study),
        "slice": opt_viz.plot_slice(study),
        "timeline": opt_viz.plot_timeline(study),
    }
    for name, fig in tqdm(figs.items(), desc="Saving Optuna visualizations"):
        try:
            fig.write_html(str(out_dir / f"{name}.html"))
            fig.write_image(str(out_dir / f"{name}.png"))
        except Exception as e:
            logger.warning(f"Could not export '{name}': {e}")
    logger.info(f"Optuna visualizations saved to {out_dir}")


if study is not None:
    save_optuna_visualizations(study)


## 17. Resolve Final Hyperparameters

In [ ]:

def load_best_params(cfg: Config = CFG) -> dict:
    best_path = cfg.OPTUNA_DIR / "best_params.json"
    if best_path.exists():
        with open(best_path) as f:
            payload = json.load(f)
        logger.info(f"Loaded best params from {best_path} (RMSE={payload['best_value']:.6f})")
        return payload["best_params"]
    logger.warning("No best_params.json found — falling back to sane hand-tuned defaults.")
    return {
        "n_cnn_layers": 4, "cnn_base_channels": 64, "cnn_kernel_size": 5, "cnn_dropout": 0.1,
        "geology_embed_dim": 8, "d_model": 128, "n_transformer_layers": 3, "n_transformer_heads": 8,
        "transformer_ff_dim": 256, "transformer_dropout": 0.1, "head_hidden_dim": 128, "head_dropout": 0.1,
        "learning_rate": 3e-4, "weight_decay": 1e-4, "batch_size": 256, "grad_clip_norm": 1.0, "window_size": 256,
    }


best_raw = load_best_params(CFG)
n_cnn_layers = best_raw["n_cnn_layers"]
cnn_channels_final = tuple(best_raw["cnn_base_channels"] * (2 ** min(i, 2)) for i in range(n_cnn_layers))

FINAL_CFG = make_fold_config(
    CFG,
    model_hp={
        "CNN_CHANNELS": cnn_channels_final,
        "CNN_KERNEL_SIZE": best_raw["cnn_kernel_size"],
        "CNN_DROPOUT": best_raw["cnn_dropout"],
        "GEOLOGY_EMBED_DIM": best_raw["geology_embed_dim"],
        "D_MODEL": best_raw["d_model"],
        "N_TRANSFORMER_LAYERS": best_raw["n_transformer_layers"],
        "N_TRANSFORMER_HEADS": best_raw["n_transformer_heads"],
        "TRANSFORMER_FF_DIM": best_raw["transformer_ff_dim"],
        "TRANSFORMER_DROPOUT": best_raw["transformer_dropout"],
        "HEAD_HIDDEN_DIM": best_raw["head_hidden_dim"],
        "HEAD_DROPOUT": best_raw["head_dropout"],
    },
    train_hp={
        "LEARNING_RATE": best_raw["learning_rate"], "WEIGHT_DECAY": best_raw["weight_decay"],
        "BATCH_SIZE": best_raw["batch_size"], "GRAD_CLIP_NORM": best_raw["grad_clip_norm"],
        "WINDOW_SIZE": best_raw["window_size"],
    },
)
FINAL_CFG.MAX_EPOCHS = CFG.FINAL_MAX_EPOCHS
FINAL_CFG.EARLY_STOPPING_PATIENCE = CFG.FINAL_EARLY_STOPPING_PATIENCE

logger.info(f"Final model config:\n{json.dumps({k: str(v) for k, v in asdict(FINAL_CFG).items()}, indent=2)}")
with open(CFG.MODELS_DIR / "final_config.json", "w") as f:
    json.dump({k: str(v) for k, v in asdict(FINAL_CFG).items()}, f, indent=2)


## 18. Final Cross-Validated Training — Window→Row OOF Reconstruction

In [ ]:

def train_cv_models(folds_df: pd.DataFrame, cfg: Config = CFG) -> Dict[str, Any]:
    row_pred_sum = np.zeros(len(folds_df), dtype=np.float64)
    row_pred_count = np.zeros(len(folds_df), dtype=np.float64)

    fold_models: Dict[int, nn.Module] = {}
    fold_scalers: Dict[int, StandardScaler] = {}
    fold_metrics: List[dict] = []
    fold_histories: Dict[int, dict] = {}

    folds_df = folds_df.reset_index(drop=False).rename(columns={"index": "_orig_index"})
    orig_index_to_pos = pd.Series(np.arange(len(folds_df)), index=folds_df["_orig_index"].values)

    for fold in tqdm(range(cfg.N_FOLDS), desc="Training CV folds"):
        t0 = time.time()
        logger.info(f"===== FOLD {fold} =====")

        tr = folds_df[folds_df.fold != fold].copy()
        va = folds_df[folds_df.fold == fold].copy()

        scaler = fit_scaler_on_fold(tr, CONTINUOUS_FEATURE_COLS)
        tr = scale_df(tr, CONTINUOUS_FEATURE_COLS, scaler)
        va_scaled = scale_df(va, CONTINUOUS_FEATURE_COLS, scaler)

        # FIX: standardize Target_TVT on this fold's training rows only, then train
        # the network on the z-scored target. Predictions get inverse-transformed
        # back to feet right after the forward pass below, so everything downstream
        # (row_pred_sum/row_pred_count, oof metrics) is already in real units.
        target_stats = fit_target_scaler(tr, cfg.TARGET_COL)
        tr = scale_target_col(tr, cfg.TARGET_COL, target_stats)
        va_scaled = scale_target_col(va_scaled, cfg.TARGET_COL, target_stats)

        train_ds = WellWindowDataset(tr, CONTINUOUS_FEATURE_COLS, cfg.WINDOW_SIZE,
                                      cfg.WINDOW_STRIDE_TRAIN, True, cfg, target_col=SCALED_TARGET_COL)
        val_ds_train_stride = WellWindowDataset(va_scaled, CONTINUOUS_FEATURE_COLS, cfg.WINDOW_SIZE,
                                                 cfg.WINDOW_STRIDE_TRAIN, True, cfg, target_col=SCALED_TARGET_COL)
        val_ds_dense = WellWindowDataset(va_scaled, CONTINUOUS_FEATURE_COLS, cfg.WINDOW_SIZE,
                                          cfg.WINDOW_STRIDE_INFER, True, cfg, target_col=SCALED_TARGET_COL)

        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                                   num_workers=cfg.NUM_WORKERS, collate_fn=collate_windows,
                                   pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds_train_stride, batch_size=cfg.BATCH_SIZE, shuffle=False,
                                 num_workers=cfg.NUM_WORKERS, collate_fn=collate_windows, pin_memory=True)
        val_loader_dense = DataLoader(val_ds_dense, batch_size=cfg.BATCH_SIZE, shuffle=False,
                                       num_workers=cfg.NUM_WORKERS, collate_fn=collate_windows, pin_memory=True)

        model = build_model(len(CONTINUOUS_FEATURE_COLS), N_GEOLOGY_CLASSES, cfg)
        model, result = train_model(
            model, train_loader, val_loader,
            max_epochs=cfg.MAX_EPOCHS, learning_rate=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY,
            grad_clip=cfg.GRAD_CLIP_NORM, patience=cfg.EARLY_STOPPING_PATIENCE, fold=fold,
            target_std=target_stats["std"],
        )

        model.eval()
        with torch.no_grad():
            for batch in val_loader_dense:
                x = batch["x"].to(DEVICE)
                geo = batch["geo"].to(DEVICE)
                mask = batch["mask"].to(DEVICE)
                row_idx = batch["row_idx"].numpy()

                with autocast(enabled=cfg.USE_AMP):
                    pred = model(x, geo, mask)
                pred = pred.float().cpu().numpy()
                pred = inverse_transform_target(pred, target_stats)  # back to real feet
                mask_np = mask.cpu().numpy()

                valid = (row_idx >= 0) & (mask_np > 0)
                flat_rows = row_idx[valid]
                flat_preds = pred[valid]
                positions = orig_index_to_pos.loc[flat_rows].values
                np.add.at(row_pred_sum, positions, flat_preds)
                np.add.at(row_pred_count, positions, 1.0)

        model_path = cfg.MODELS_DIR / f"cnn_transformer_fold{fold}.pt"
        torch.save({"model_state_dict": model.state_dict(),
                    "config": {k: str(v) for k, v in asdict(cfg).items()}}, model_path)

        fold_models[fold] = model
        fold_scalers[fold] = {"feature_scaler": scaler, "target_mean": target_stats["mean"],
                               "target_std": target_stats["std"]}
        fold_histories[fold] = result["history"]
        fold_metrics.append({
            "fold": fold, "best_val_rmse": result["best_val_rmse"],
            "n_epochs_trained": len(result["history"]["val_rmse"]),
            "n_train_wells": tr[cfg.GROUP_COL].nunique(), "n_val_wells": va[cfg.GROUP_COL].nunique(),
            "train_time_s": time.time() - t0,
        })
        logger.info(f"Fold {fold} | best_val_rmse={result['best_val_rmse']:.5f} | saved -> {model_path}")

        del train_loader, val_loader, val_loader_dense
        torch.cuda.empty_cache()
        gc.collect()

    valid_rows = row_pred_count > 0
    oof_pred = np.full(len(folds_df), np.nan, dtype=np.float32)
    oof_pred[valid_rows] = (row_pred_sum[valid_rows] / row_pred_count[valid_rows]).astype(np.float32)
    folds_df["oof_pred"] = oof_pred

    valid_mask = folds_df["oof_pred"].notna()
    overall_rmse = math.sqrt(mean_squared_error(folds_df.loc[valid_mask, cfg.TARGET_COL], folds_df.loc[valid_mask, "oof_pred"]))
    overall_mae = mean_absolute_error(folds_df.loc[valid_mask, cfg.TARGET_COL], folds_df.loc[valid_mask, "oof_pred"])
    overall_r2 = r2_score(folds_df.loc[valid_mask, cfg.TARGET_COL], folds_df.loc[valid_mask, "oof_pred"])

    logger.info(f"===== OVERALL OOF METRICS (row-level, overlap-averaged) =====")
    logger.info(f"RMSE={overall_rmse:.5f} MAE={overall_mae:.5f} R2={overall_r2:.5f} "
                f"(coverage={valid_mask.mean()*100:.2f}%)")

    return {
        "oof_df": folds_df, "fold_models": fold_models, "fold_scalers": fold_scalers,
        "fold_metrics": pd.DataFrame(fold_metrics), "fold_histories": fold_histories,
        "overall_rmse": overall_rmse, "overall_mae": overall_mae, "overall_r2": overall_r2,
    }


cv_results = train_cv_models(train_df, FINAL_CFG)
cv_results["fold_metrics"].to_csv(CFG.LOGS_DIR / "fold_metrics.csv", index=False)
cv_results["fold_metrics"]


## 19. CV Metric Summary & CSV Logging

In [ ]:

summary_row = {
    "run_id": RUN_ID, "timestamp": datetime.now().isoformat(),
    "overall_oof_rmse": cv_results["overall_rmse"], "overall_oof_mae": cv_results["overall_mae"],
    "overall_oof_r2": cv_results["overall_r2"],
    "mean_fold_rmse": cv_results["fold_metrics"]["best_val_rmse"].mean(),
    "std_fold_rmse": cv_results["fold_metrics"]["best_val_rmse"].std(),
    "n_folds": CFG.N_FOLDS,
    "final_config": json.dumps({k: str(v) for k, v in asdict(FINAL_CFG).items()}),
}

run_log_path = CFG.LOGS_DIR / "run_history.csv"
run_log_df = pd.DataFrame([summary_row])
if run_log_path.exists():
    run_log_df = pd.concat([pd.read_csv(run_log_path), run_log_df], ignore_index=True)
run_log_df.to_csv(run_log_path, index=False)

logger.info(f"Run summary appended to {run_log_path}")
logger.info(f"Mean fold RMSE: {summary_row['mean_fold_rmse']:.5f} +/- {summary_row['std_fold_rmse']:.5f}")
logger.info(f"Overall OOF RMSE: {summary_row['overall_oof_rmse']:.5f}")


## 20. Self-Attention Visualization

Plots the final Transformer layer's self-attention weight matrix for a sample window,
showing which positions along the well the model treats as most relevant to each other.

In [ ]:

def plot_attention_sample(model: nn.Module, dataset: WellWindowDataset, cfg: Config = CFG,
                           sample_idx: int = 0, out_path: Optional[Path] = None) -> None:
    model.eval()
    item = dataset[sample_idx]
    with torch.no_grad():
        x = item["x"].unsqueeze(0).to(DEVICE)
        geo = item["geo"].unsqueeze(0).to(DEVICE)
        mask = item["mask"].unsqueeze(0).to(DEVICE)
        with autocast(enabled=cfg.USE_AMP):
            _, attn_weights = model(x, geo, mask, return_attention=True)

    attn = attn_weights[0].float().cpu().numpy()
    n_valid = int(item["mask"].sum().item())

    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(attn[:n_valid, :n_valid], aspect="auto", cmap="viridis")
    ax.set_xlabel("Window position (key)")
    ax.set_ylabel("Window position (query)")
    ax.set_title("Transformer Self-Attention (final layer)")
    plt.colorbar(im, ax=ax, label="Attention weight")
    plt.tight_layout()
    if out_path:
        fig.savefig(out_path)
    plt.close(fig)


fold0_val = train_df[train_df.fold == 0].copy()
scaler0 = cv_results["fold_scalers"][0]["feature_scaler"]
fold0_val_scaled = scale_df(fold0_val, CONTINUOUS_FEATURE_COLS, scaler0)
viz_ds = WellWindowDataset(fold0_val_scaled, CONTINUOUS_FEATURE_COLS, FINAL_CFG.WINDOW_SIZE,
                            FINAL_CFG.WINDOW_STRIDE_TRAIN, True, FINAL_CFG)

for i in range(min(3, len(viz_ds))):
    plot_attention_sample(cv_results["fold_models"][0], viz_ds, FINAL_CFG, sample_idx=i,
                           out_path=CFG.ATTENTION_DIR / f"self_attention_sample{i}.png")

logger.info(f"Self-attention visualizations saved to {CFG.ATTENTION_DIR}")


## 21. Prediction Analysis & Residual Diagnostics

In [ ]:

def plot_prediction_analysis(oof_df: pd.DataFrame, cfg: Config = CFG) -> None:
    valid = oof_df["oof_pred"].notna()
    y_true = oof_df.loc[valid, cfg.TARGET_COL].values
    y_pred = oof_df.loc[valid, "oof_pred"].values
    residuals = y_true - y_pred

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    ax = axes[0, 0]
    sample_idx = np.random.default_rng(cfg.SEED).choice(len(y_true), size=min(100000, len(y_true)), replace=False)
    ax.scatter(y_true[sample_idx], y_pred[sample_idx], s=2, alpha=0.15, color="#2E86AB")
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="y = x")
    ax.set_xlabel("Actual TVT"); ax.set_ylabel("Predicted TVT")
    ax.set_title("Predicted vs. Actual (OOF)"); ax.legend()

    ax = axes[0, 1]
    ax.scatter(y_pred[sample_idx], residuals[sample_idx], s=2, alpha=0.15, color="#A23B72")
    ax.axhline(0, color="r", linestyle="--", linewidth=1.5)
    ax.set_xlabel("Predicted TVT"); ax.set_ylabel("Residual (Actual - Predicted)")
    ax.set_title("Residuals vs. Predicted")

    ax = axes[1, 0]
    sns.histplot(residuals, bins=100, kde=True, ax=ax, color="#F18F01")
    ax.axvline(0, color="r", linestyle="--")
    ax.set_xlabel("Residual"); ax.set_title(f"Residual Distribution (skew={stats.skew(residuals):.3f})")

    ax = axes[1, 1]
    well_rmse = oof_df.loc[valid].groupby(cfg.GROUP_COL).apply(
        lambda g: math.sqrt(mean_squared_error(g[cfg.TARGET_COL], g["oof_pred"])))
    sns.histplot(well_rmse, bins=50, ax=ax, color="#3B1F2B")
    ax.set_xlabel("Per-Well RMSE"); ax.set_title("Distribution of Per-Well OOF RMSE")

    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / "prediction_analysis.png")
    plt.close(fig)

    well_rmse.sort_values(ascending=False).to_csv(cfg.FIGURES_DIR / "per_well_rmse.csv")
    logger.info(f"Worst 10 wells by OOF RMSE:\n{well_rmse.sort_values(ascending=False).head(10).to_string()}")
    logger.info(f"Best 10 wells by OOF RMSE:\n{well_rmse.sort_values().head(10).to_string()}")


plot_prediction_analysis(cv_results["oof_df"], CFG)


## 22. Learning Curves

In [ ]:

def plot_learning_curves(fold_histories: Dict[int, dict], cfg: Config = CFG) -> None:
    fig, ax = plt.subplots(figsize=(11, 7))
    colors = sns.color_palette("husl", len(fold_histories))
    for (fold, hist), color in zip(fold_histories.items(), colors):
        epochs = np.arange(len(hist["val_rmse"]))
        ax.plot(epochs, np.sqrt(hist["train_loss"]), color=color, linestyle="--", alpha=0.6, label=f"fold{fold} train")
        ax.plot(epochs, hist["val_rmse"], color=color, linestyle="-", linewidth=2, label=f"fold{fold} val")
    ax.set_xlabel("Epoch"); ax.set_ylabel("RMSE")
    ax.set_title("Learning Curves — Train vs. Validation RMSE per Fold")
    ax.legend(ncol=2, fontsize=8)
    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / "learning_curves.png")
    plt.close(fig)
    logger.info(f"Learning curves saved to {cfg.FIGURES_DIR / 'learning_curves.png'}")


plot_learning_curves(cv_results["fold_histories"], CFG)


## 23. Model, Scaler & Config Persistence

In [ ]:

def persist_run_artifacts(cv_results: dict, cfg: Config = CFG) -> None:
    with open(cfg.MODELS_DIR / "fold_scalers.pkl", "wb") as f:
        pickle.dump(cv_results["fold_scalers"], f)

    manifest = {
        "run_id": RUN_ID, "n_folds": cfg.N_FOLDS,
        "overall_oof_rmse": cv_results["overall_rmse"], "overall_oof_mae": cv_results["overall_mae"],
        "overall_oof_r2": cv_results["overall_r2"],
        "fold_model_paths": [str(cfg.MODELS_DIR / f"cnn_transformer_fold{f}.pt")
                              for f in cv_results["fold_models"]],
        "continuous_feature_cols": CONTINUOUS_FEATURE_COLS,
        "n_geology_classes": int(N_GEOLOGY_CLASSES),
        "created_at": datetime.now().isoformat(),
    }
    with open(cfg.MODELS_DIR / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)
    logger.info(f"All model artifacts persisted to {cfg.MODELS_DIR}")


persist_run_artifacts(cv_results, CFG)
cv_results["oof_df"][[CFG.GROUP_COL, CFG.MD_COL, CFG.TARGET_COL, "oof_pred", "fold"]].to_csv(
    CFG.LOGS_DIR / "oof_predictions.csv", index=False
)


## 24. Inference Pipeline

Runs dense-stride windowed inference on the test set with every fold model, reconstructs
row-level predictions via overlap-averaging, then averages across fold models.

In [ ]:
print(cv_results.keys())

In [ ]:

class InferencePipeline:
    def __init__(self, cfg: Config, fold_models: Dict[int, nn.Module],
                 fold_scalers: Dict[int, dict]):
        self.cfg = cfg
        self.fold_models = fold_models
        self.fold_scalers = fold_scalers  # each value: {"feature_scaler", "target_mean", "target_std"}

    def _predict_fold(self, model: nn.Module, df: pd.DataFrame, scaler_bundle: dict) -> np.ndarray:
        feature_scaler = scaler_bundle["feature_scaler"]
        target_stats = {"mean": scaler_bundle["target_mean"], "std": scaler_bundle["target_std"]}
        df_scaled = scale_df(df, CONTINUOUS_FEATURE_COLS, feature_scaler)
        ds = WellWindowDataset(df_scaled, CONTINUOUS_FEATURE_COLS,
                                self.cfg.WINDOW_SIZE, self.cfg.WINDOW_STRIDE_INFER,
                                has_target=False, cfg=self.cfg)
        loader = DataLoader(ds, batch_size=self.cfg.BATCH_SIZE, shuffle=False,
                     num_workers=0, collate_fn=collate_windows, pin_memory=True)

        pred_sum = np.zeros(len(df), dtype=np.float64)
        pred_count = np.zeros(len(df), dtype=np.float64)
        orig_index_to_pos = pd.Series(np.arange(len(df)), index=df.index.values)

        model.eval()
        with torch.no_grad():
            for batch in tqdm(loader, desc="Fold inference", leave=False):
                x = batch["x"].to(DEVICE); geo = batch["geo"].to(DEVICE); mask = batch["mask"].to(DEVICE)
                row_idx = batch["row_idx"].numpy()
                with autocast(enabled=self.cfg.USE_AMP):
                    pred = model(x, geo, mask)
                pred = pred.float().cpu().numpy()
                pred = inverse_transform_target(pred, target_stats)  # back to real feet
                mask_np = mask.cpu().numpy()
                valid = (row_idx >= 0) & (mask_np > 0)
                flat_rows = row_idx[valid]
                flat_preds = pred[valid]
                positions = orig_index_to_pos.loc[flat_rows].values
                np.add.at(pred_sum, positions, flat_preds)
                np.add.at(pred_count, positions, 1.0)

        pred_count = np.maximum(pred_count, 1.0)
        return (pred_sum / pred_count).astype(np.float32)

    def predict(self, raw_test_df: pd.DataFrame) -> pd.DataFrame:
        df = raw_test_df.copy().reset_index(drop=True)
        logger.info(f"Running inference on {len(df):,} test rows across {df[self.cfg.GROUP_COL].nunique()} wells")

        fold_preds = np.zeros((len(df), len(self.fold_models)), dtype=np.float32)
        for i, (fold, model) in enumerate(tqdm(self.fold_models.items(), desc="Fold ensemble inference")):
            fold_preds[:, i] = self._predict_fold(model, df, self.fold_scalers[fold])

        df["prediction"] = fold_preds.mean(axis=1)
        df["prediction_std"] = fold_preds.std(axis=1)
        return df


if test_df is not None:
    inference = InferencePipeline(CFG, cv_results["fold_models"], cv_results["fold_scalers"])
    test_pred_df = inference.predict(test_df)
    logger.info(f"Inference complete. Prediction stats:\n{test_pred_df['prediction'].describe()}")
else:
    logger.warning("No test set available — skipping inference cell.")
    test_pred_df = None


In [ ]:
sample_sub = pd.read_csv(CFG.DATA_DIR / CFG.SAMPLE_SUB_FILE)
print("sample_submission shape:", sample_sub.shape)
print("\nFirst 10 ids:")
print(sample_sub["id"].head(10).tolist())
print("\nLast 5 ids:")
print(sample_sub["id"].tail(5).tolist())

print("\n--- test_pred_df ---")
print("shape:", test_pred_df.shape)
print("unique WELL_NAME count:", test_pred_df["WELL_NAME"].nunique())
print("sample WELL_NAME values:", test_pred_df["WELL_NAME"].unique()[:5])
print("MD range:", test_pred_df["MD"].min(), "-", test_pred_df["MD"].max())

# rows per well, both sides
print("\nrows per well in test_pred_df (first 5 wells):")
print(test_pred_df.groupby("WELL_NAME").size().head())

In [ ]:
# Build the candidate id using per-well positional index in MD-ascending order
test_pred_df_sorted = test_pred_df.sort_values(["WELL_NAME", "MD"]).reset_index(drop=True)
test_pred_df_sorted["pos_in_well"] = test_pred_df_sorted.groupby("WELL_NAME").cumcount()
test_pred_df_sorted["id"] = test_pred_df_sorted["WELL_NAME"] + "_" + test_pred_df_sorted["pos_in_well"].astype(str)

sample_sub = pd.read_csv(CFG.DATA_DIR / CFG.SAMPLE_SUB_FILE)

matched = sample_sub["id"].isin(test_pred_df_sorted["id"])
print(f"Matched {matched.sum()} / {len(sample_sub)} sample_submission ids")

# spot-check the well that starts mid-way (000d7d20)
print(test_pred_df_sorted[test_pred_df_sorted.WELL_NAME == "000d7d20"][["pos_in_well","MD"]].iloc[[0, 1441, 1442, 1443]])
print(sample_sub[sample_sub["id"].str.startswith("000d7d20")].head(3))

## 25. Submission File Generation

In [ ]:
def build_submission(test_pred_df: pd.DataFrame, cfg: Config = CFG,
                      sample_sub_path: Optional[Path] = None) -> pd.DataFrame:
    sample_sub_path = sample_sub_path or (cfg.DATA_DIR / cfg.SAMPLE_SUB_FILE)
    sample_sub = pd.read_csv(sample_sub_path)

    target_col_name = [c for c in sample_sub.columns if "tvt" in c.lower() or c.lower() == "target"]
    target_col_name = target_col_name[0] if target_col_name else sample_sub.columns[-1]
    id_col = [c for c in sample_sub.columns if c != target_col_name][0]

    df = test_pred_df.sort_values([cfg.GROUP_COL, cfg.MD_COL]).reset_index(drop=True)
    df["pos_in_well"] = df.groupby(cfg.GROUP_COL).cumcount()
    df["_id"] = df[cfg.GROUP_COL].astype(str) + "_" + df["pos_in_well"].astype(str)

    merged = sample_sub[[id_col]].merge(
        df[["_id", "prediction"]], left_on=id_col, right_on="_id", how="left"
    )
    n_missing = merged["prediction"].isna().sum()
    if n_missing > 0:
        logger.warning(f"{n_missing} sample_submission ids had no matching prediction — "
                        f"check the id-index convention (0-based vs 1-based, sort order).")

    submission = merged[[id_col]].copy()
    submission[target_col_name] = merged["prediction"].values

    out_path = cfg.SUBMISSIONS_DIR / f"submission_{RUN_ID}.csv"
    submission.to_csv(out_path, index=False)
    submission.to_csv(cfg.SUBMISSIONS_DIR / "submission_latest.csv", index=False)
    logger.info(f"Submission saved -> {out_path}  (shape={submission.shape}, missing={n_missing})")
    return submission


if test_pred_df is not None:
    submission_df = build_submission(test_pred_df, CFG)
    submission_df.head()
    print(submission_df["tvt"].describe())
    print(submission_df["tvt"].isna().sum())
else:
    logger.info("Submission generation skipped (no test set loaded).")

## 26. Final Run Summary

In [ ]:

final_summary = {
    "run_id": RUN_ID,
    "architecture": "CNN + Transformer",
    "n_train_rows": int(len(train_df)), "n_train_wells": int(train_df[CFG.GROUP_COL].nunique()),
    "n_continuous_features": len(CONTINUOUS_FEATURE_COLS), "n_geology_classes": int(N_GEOLOGY_CLASSES),
    "n_folds": CFG.N_FOLDS,
    "overall_oof_rmse": cv_results["overall_rmse"], "overall_oof_mae": cv_results["overall_mae"],
    "overall_oof_r2": cv_results["overall_r2"],
    "per_fold_rmse": cv_results["fold_metrics"][["fold", "best_val_rmse", "n_epochs_trained"]].to_dict(orient="records"),
    "final_config": {k: str(v) for k, v in asdict(FINAL_CFG).items()},
    "optuna_trials_run": len(study.trials) if study is not None else "skipped",
    "optuna_best_value": study.best_value if study is not None and study.best_trial else None,
    "artifact_dirs": {
        "models": str(CFG.MODELS_DIR), "logs": str(CFG.LOGS_DIR), "figures": str(CFG.FIGURES_DIR),
        "optuna": str(CFG.OPTUNA_DIR), "submissions": str(CFG.SUBMISSIONS_DIR), "attention": str(CFG.ATTENTION_DIR),
    },
    "completed_at": datetime.now().isoformat(),
}

with open(CFG.LOGS_DIR / f"run_summary_{RUN_ID}.json", "w") as f:
    json.dump(final_summary, f, indent=2, default=str)

logger.info("=" * 80)
logger.info("RUN COMPLETE")
logger.info(f"Overall OOF RMSE : {final_summary['overall_oof_rmse']:.6f}")
logger.info(f"Overall OOF MAE  : {final_summary['overall_oof_mae']:.6f}")
logger.info(f"Overall OOF R2   : {final_summary['overall_oof_r2']:.6f}")
logger.info(f"Continuous feats : {final_summary['n_continuous_features']}")
logger.info(f"Artifacts root   : {CFG.OUTPUT_DIR.resolve()}")
logger.info("=" * 80)

print(json.dumps(final_summary, indent=2, default=str))
